# Notebook 1 — Anatomie d'un appel LLM robuste : ce qui casse en prod, et comment on l'encaisse

Ce notebook lit et exécute la **couche LLM** du banc d'essai (`src/llm_testbench/llm/`, ADR 006).
Il n'implémente rien : chaque bloc de code est importé depuis `src/` et exécuté tel quel.

**Ce que vous saurez à la fin** : pourquoi un appel LLM « qui marche » n'est pas un appel robuste,
ce qui casse réellement en production (limites de débit, pannes, délais, JSON invalide, coûts,
données qui partent où il ne faut pas), et comment chaque maillon de notre couche l'encaisse —
en regardant à chaque fois **ce qui part sur le réseau** et **ce qui revient**.

**Comment lire** : les cellules de configuration sont en haut ; les scénarios de panne sont joués
sur un **réseau simulé** (le vrai SDK, de fausses réponses HTTP), donc ils tournent hors ligne, en
CI comme chez vous. La dernière partie exécute un vrai appel si une clé est disponible.

> Vocabulaire minimal, défini au fil de l'eau ; lexique récapitulatif en fin de notebook.

## Partie 1 — Le problème

Un **appel LLM** est une requête HTTP vers un service distant (le **provider** : Anthropic, OpenAI…)
qui reçoit un **prompt** (du texte, découpé en **tokens**, l'unité de facturation) et renvoie une
**complétion** (du texte généré, lui aussi compté en tokens). Rien de plus exotique qu'un appel
d'API — sauf que le service est lent (secondes), cher (facturé au token), non déterministe, et
qu'il tombe ou vous limite plus souvent qu'une base de données.

L'appel naïf tient en cinq lignes avec le SDK officiel. Et voici ce qui lui arrive en production :

| Ce qui casse | Symptôme | Ce qu'il faut |
|---|---|---|
| Limite de débit (HTTP 429) | « too many requests », parfois avec un délai conseillé | réessayer, après le délai conseillé |
| Provider surchargé ou en panne (529, 5xx, coupure) | erreurs en rafale | réessayer un peu, puis **arrêter d'insister** (circuit breaker) |
| Réponse qui ne vient pas | l'appel pend | un **délai** au-delà duquel c'est une panne |
| Réponse tronquée | texte coupé au milieu | lire la **raison de fin** (`length`), pas seulement le texte |
| JSON demandé, JSON invalide reçu | parsing qui explose | valider, et **réparer** en renvoyant l'erreur au modèle |
| Coût qui dérape | facture surprise | un **budget** vérifié avant l'appel, un coût calculé après |
| La même question mille fois | mille fois le prix | un **cache** exact, puis sémantique |
| Des données envoyées au mauvais endroit | incident de conformité | une **politique de gouvernance** vérifiée avant tout appel |

Chaque ligne de ce tableau est un module de `src/llm_testbench/llm/`. On va les lire, puis les faire
tomber un par un.

In [1]:
# --- Configuration : modifiez, puis « Run All » ---
import os
from pathlib import Path

# Mode échantillon (CI) : réseau simulé uniquement. LLM_TESTBENCH_SAMPLE=1 l'active.
SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"

# Charge .env dans l'environnement (clés API) sans dépendance supplémentaire.
for line in (
    (Path.cwd().parent / ".env").read_text().splitlines()
    if (Path.cwd().parent / ".env").is_file()
    else []
):
    if "=" in line and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

# Le vrai appel (partie 4.9) n'a lieu qu'en mode complet ET avec une clé.
USE_REAL_API = (not SAMPLE) and bool(os.environ.get("ANTHROPIC_API_KEY"))

# Modèle « backend/modèle » (config/llm_backends.toml). Essayez "openai/gpt-5.6-luna".
MODEL = "anthropic/claude-sonnet-5"

# Boutons des expérimentations (partie 6) :
RETRY_MAX_ATTEMPTS = 3  # tentatives maximales par appel
BREAKER_THRESHOLD = 2  # échecs consécutifs avant ouverture du circuit
TIMEOUT_S = 0.2  # délai maximal d'un appel (les scénarios simulent des réponses lentes)
SEMANTIC_THRESHOLD = 0.95  # similarité minimale pour un coup de cache sémantique
DATA_CLASS = "public"  # essayez "regulated" pour voir la gouvernance refuser

print(f"mode échantillon : {SAMPLE} — vrai appel API : {USE_REAL_API} — modèle : {MODEL}")

mode échantillon : True — vrai appel API : False — modèle : anthropic/claude-sonnet-5


In [2]:
import inspect
import json
import random
import time

from IPython.display import Markdown, display
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from llm_testbench.config import find_repo_root
from llm_testbench.llm import CompletionRequest, DataClass, Message, ModelRef, Role, Usage
from llm_testbench.llm.budget import TokenBudget
from llm_testbench.llm.cache import InMemoryCache, exact_key, semantic_namespace
from llm_testbench.llm.client import LLMClient, SemanticCacheConfig
from llm_testbench.llm.errors import (
    BudgetExceededError,
    GovernanceViolation,
    LLMError,
    ProviderTimeoutError,
)
from llm_testbench.llm.governance import GovernancePolicy
from llm_testbench.llm.prompts import PromptRegistry
from llm_testbench.llm.providers.anthropic import AnthropicProvider
from llm_testbench.llm.resilience import CircuitBreaker, RetryPolicy
from llm_testbench.llm.structured import StructuredOutputs
from llm_testbench.llm.testing import (
    ScriptedTransport,
    Step,
    ToyEmbedder,
    anthropic_error,
    anthropic_message,
    anthropic_sse,
    scripted_anthropic_provider,
)
from llm_testbench.obs.tracing import configure_tracing, get_tracer

ROOT = find_repo_root()


def show_source(*objects: object) -> None:
    """Affiche la source réelle d'objets de src/ (pur affichage, aucune logique ici)."""
    display(Markdown("\n".join(f"```python\n{inspect.getsource(o)}```" for o in objects)))


def show_span(exporter: InMemorySpanExporter, prefixes=("gen_ai.", "llm_testbench.")) -> None:
    """Affiche les attributs du dernier span exporté (ce qu'un backend de traces recevrait)."""
    span = exporter.get_finished_spans()[-1]
    attrs = dict(span.attributes or {})
    print(f"span « {span.name} » — statut {span.status.status_code.name}")
    for key in sorted(k for k in attrs if k.startswith(prefixes)):
        print(f"  {key:<46} {attrs[key]!r}")
    exporter.clear()


def ask(
    question: str, *, system: str = "Réponds en une phrase.", data_class: str = DATA_CLASS, **kw
) -> CompletionRequest:
    return CompletionRequest(
        model=ModelRef.parse(MODEL),
        messages=(
            Message(role=Role.SYSTEM, content=system),
            Message(role=Role.USER, content=question),
        ),
        data_class=DataClass(data_class),
        max_output_tokens=200,
        **kw,
    )


print(f"racine du dépôt : {ROOT}")

racine du dépôt : /Users/christophe/Roadmaps/roadmap-ai-engineer


### Exemple jouet : l'appel naïf, et ce qu'il fait quand le provider est surchargé

Avant notre couche, regardons le SDK seul. On lui branche un **réseau simulé** : la requête part
vraiment dans le SDK (construction du corps, en-têtes, sérialisation), mais la réponse HTTP est
celle qu'on a décidée — ici un **529 « overloaded »**, l'erreur qu'Anthropic renvoie en période de
pointe. Le SDK retente de lui-même deux fois par défaut ; on le désactive (`max_retries=0`) pour
voir l'erreur brute, et parce que notre couche gère les retries elle-même (partie 3.4).

In [3]:
import anthropic

script = ScriptedTransport([anthropic_error(529, "Overloaded")])
naive_client = anthropic.AsyncAnthropic(
    api_key="fake-key",
    max_retries=0,
    http_client=anthropic.DefaultAsyncHttpxClient(transport=script.transport()),
)
try:
    await naive_client.messages.create(
        model="claude-sonnet-5",
        max_tokens=100,
        messages=[{"role": "user", "content": "Quelle est la capitale de la France ?"}],
    )
except Exception as error:
    print(f"exception brute du SDK : {type(error).__module__}.{type(error).__name__}")
    print(f"message : {error}")

print("\ncorps réellement envoyé par le SDK :")
print(json.dumps(script.requests[0], indent=2, ensure_ascii=False))

exception brute du SDK : anthropic.OverloadedError
message : Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error', 'message': 'Overloaded'}}

corps réellement envoyé par le SDK :
{
  "max_tokens": 100,
  "messages": [
    {
      "role": "user",
      "content": "Quelle est la capitale de la France ?"
    }
  ],
  "model": "claude-sonnet-5"
}


Le SDK a fait son travail : il a levé une exception typée. Mais l'appel naïf, lui, s'arrête là :
pas de nouvelle tentative, pas de délai, pas de compteur d'échecs, pas de trace, pas de coût.
Tout ce qui suit consiste à *entourer* ce SDK, sans le remplacer.

## Partie 2 — Les options

| Option | Ce qu'elle apporte | Pourquoi pas (ADR 006) |
|---|---|---|
| **LiteLLM** ou équivalent | cent providers derrière une API unique | masque précisément la mécanique qu'on veut apprendre ; dépendance lourde |
| **OpenRouter** comme chemin unique | un seul client compatible OpenAI, catalogue immense | intermédiaire de plus pour la gouvernance des données, pas d'API batch, comptage de tokens normalisé |
| **Notre couche sur les SDK officiels** ✔ | deux implémentations (Anthropic natif, compatible OpenAI) pour six backends ; chaque maillon lisible et testé | plus de code à nous — c'est le but |

OpenRouter reste un *backend de largeur* (Mistral, Qwen, Llama sans clé dédiée), jamais le chemin
unique ; Bedrock (résidence UE) passe par le client Bedrock du SDK Anthropic ; vLLM (phase 6) par
le client compatible OpenAI.

## Partie 3 — Lecture guidée du code réel

### 3.1 La requête neutre : indépendante du SDK, figée, et gouvernée

Tout commence par `CompletionRequest`. Deux choix à remarquer : elle est **figée** (`frozen`) — on
ne la modifie pas, on la dérive — et sa **classe de données** (`data_class`) est **obligatoire**,
sans valeur par défaut : le défaut implicite serait la première faille de gouvernance.

In [4]:
show_source(CompletionRequest, Usage)
request = ask("Quelle est la capitale de la France ?")
print(json.dumps(request.model_dump(mode="json"), indent=2, ensure_ascii=False))

```python
class CompletionRequest(BaseModel):
    model_config = ConfigDict(frozen=True)

    model: ModelRef
    messages: tuple[Message, ...] = Field(min_length=1)
    data_class: DataClass
    """Obligatoire, sans défaut : le défaut implicite serait la première faille."""

    max_output_tokens: int = Field(default=1024, ge=1)
    temperature: float | None = Field(default=None, ge=0.0, le=2.0)
    """``None`` : défaut du provider. ``0`` ne garantit pas le déterminisme (phase 5B)."""

    stop: tuple[str, ...] = ()
    seed: int | None = None
    json_schema: dict[str, Any] | None = None
    """Schéma JSON attendu en sortie. Le provider l'impose nativement s'il sait, sinon la
    façade l'injecte dans le prompt et valide après coup (``structured.py``)."""

    reasoning_effort: Literal["low", "medium", "high"] | None = None
    """Effort de raisonnement, quand le backend a ce réglage (``output_config.effort`` chez
    Anthropic, ``reasoning_effort`` chez OpenAI). ``None`` : défaut du provider."""

    timeout_s: float | None = Field(default=None, gt=0)
    metadata: dict[str, str] = Field(default_factory=dict)
    """Informations de trace (nom/version/empreinte du prompt…). Hors clé de cache."""

    @field_validator("messages")
    @classmethod
    def _system_only_first(cls, messages: tuple[Message, ...]) -> tuple[Message, ...]:
        for index, message in enumerate(messages):
            if message.role is Role.SYSTEM and index != 0:
                raise ValueError("un message system n'est admis qu'en première position")
        return messages

    @property
    def system(self) -> str | None:
        first = self.messages[0]
        return first.content if first.role is Role.SYSTEM else None

    @property
    def turns(self) -> tuple[Message, ...]:
        """Les messages hors system, dans l'ordre."""
        return tuple(m for m in self.messages if m.role is not Role.SYSTEM)

    def with_messages(self, messages: tuple[Message, ...]) -> "CompletionRequest":
        return self.model_copy(update={"messages": messages})
```
```python
class Usage(BaseModel):
    """Tokens consommés, normalisés entre providers.

    ``input_tokens`` compte les tokens d'entrée **non servis par le cache** (facturés plein
    tarif). Anthropic les rapporte ainsi nativement ; OpenAI inclut les tokens cachés dans
    ``prompt_tokens``, et le provider fait la soustraction. Sans cette normalisation, le
    coût par requête de la phase 5 comparerait des choses différentes.
    """

    model_config = ConfigDict(frozen=True)

    input_tokens: int = Field(default=0, ge=0)
    output_tokens: int = Field(default=0, ge=0)
    cache_read_tokens: int = Field(default=0, ge=0)
    cache_write_tokens: int = Field(default=0, ge=0)

    @property
    def total_input_tokens(self) -> int:
        return self.input_tokens + self.cache_read_tokens + self.cache_write_tokens

    @property
    def total_tokens(self) -> int:
        return self.total_input_tokens + self.output_tokens

    def __add__(self, other: "Usage") -> "Usage":
        return Usage(
            input_tokens=self.input_tokens + other.input_tokens,
            output_tokens=self.output_tokens + other.output_tokens,
            cache_read_tokens=self.cache_read_tokens + other.cache_read_tokens,
            cache_write_tokens=self.cache_write_tokens + other.cache_write_tokens,
        )
```

{
  "model": {
    "backend": "anthropic",
    "model": "claude-sonnet-5"
  },
  "messages": [
    {
      "role": "system",
      "content": "Réponds en une phrase."
    },
    {
      "role": "user",
      "content": "Quelle est la capitale de la France ?"
    }
  ],
  "data_class": "public",
  "max_output_tokens": 200,
  "temperature": null,
  "stop": [],
  "seed": null,
  "json_schema": null,
  "reasoning_effort": null,
  "timeout_s": null,
  "metadata": {}
}


### 3.2 La traduction vers le SDK, et les erreurs typées

Le **provider** traduit la requête neutre en paramètres du SDK. `build_params` est publique
exprès : on peut regarder ce qui va partir *avant* que ça parte. Deux traductions surprennent :

- `temperature` et `seed` **disparaissent** — les modèles Claude actuels (Sonnet 5, Opus 5) refusent
  tout paramètre d'échantillonnage avec un 400. Le réflexe « température 0 pour le déterminisme »
  est un réflexe de 2024 ;
- le system prompt est marqué **cacheable** (`cache_control`) : Anthropic ne cache que ce qu'on
  lui demande de cacher, et seulement au-delà d'une taille minimale (partie 4.9).

Puis `translate_error` convertit les exceptions du SDK en **erreurs typées de notre couche**, avec
un drapeau `retryable` : c'est lui, et lui seul, que le retry consultera.

In [5]:
show_source(AnthropicProvider.build_params, AnthropicProvider.translate_error)

provider, script = scripted_anthropic_provider([Step(json_body=anthropic_message())])
params = provider.build_params(ask("Bonjour", temperature=0.0, seed=42, reasoning_effort="low"))
print("paramètres SDK construits pour la requête (temperature et seed absents, effort présent) :")
print(json.dumps(params, indent=2, ensure_ascii=False))

```python
    def build_params(self, request: CompletionRequest) -> dict[str, Any]:
        """La requête neutre → les paramètres du SDK. Public : le notebook l'affiche."""
        params: dict[str, Any] = {
            "model": request.model.model,
            "max_tokens": request.max_output_tokens,
            "messages": [{"role": m.role.value, "content": m.content} for m in request.turns],
        }
        if request.system is not None:
            block: dict[str, Any] = {"type": "text", "text": request.system}
            if self.cache_system_prompt:
                block["cache_control"] = {"type": "ephemeral"}
            params["system"] = [block]
        if request.stop:
            params["stop_sequences"] = list(request.stop)
        output_config: dict[str, Any] = {}
        if request.reasoning_effort is not None:
            output_config["effort"] = request.reasoning_effort
        if request.json_schema is not None and self.native_json_schema:
            output_config["format"] = {"type": "json_schema", "schema": request.json_schema}
        if output_config:
            params["output_config"] = output_config
        if request.timeout_s is not None:
            params["timeout"] = request.timeout_s
        return params
```
```python
    def translate_error(self, error: Exception) -> LLMError:
        backend = self.backend
        if isinstance(error, anthropic.RateLimitError):
            return RateLimitedError(
                str(error),
                backend=backend,
                retry_after_s=retry_after_seconds(error.response.headers),
            )
        if isinstance(error, anthropic.APITimeoutError):
            return ProviderTimeoutError(str(error), backend=backend)
        if isinstance(error, anthropic.APIConnectionError):
            return ProviderUnavailableError(str(error), backend=backend)
        if isinstance(error, anthropic.APIStatusError):
            if error.status_code >= 500:
                return ProviderUnavailableError(
                    str(error), backend=backend, status_code=error.status_code
                )
            return InvalidRequestError(str(error), backend=backend, status_code=error.status_code)
        return ProviderUnavailableError(f"erreur inattendue : {error!r}", backend=backend)
```

paramètres SDK construits pour la requête (temperature et seed absents, effort présent) :
{
  "model": "claude-sonnet-5",
  "max_tokens": 200,
  "messages": [
    {
      "role": "user",
      "content": "Bonjour"
    }
  ],
  "system": [
    {
      "type": "text",
      "text": "Réponds en une phrase.",
      "cache_control": {
        "type": "ephemeral"
      }
    }
  ],
  "output_config": {
    "effort": "low"
  }
}


### 3.3 Retry, circuit breaker, délai : trois questions différentes

- **Retry** — « cette tentative a échoué, la suivante peut-elle réussir ? » Oui pour un 429, un
  5xx, une coupure ; non pour un 400. Le délai entre tentatives croît (**backoff exponentiel**)
  avec une **gigue** aléatoire : sans elle, tous les clients réessaient à la même seconde et
  recréent la panne. Un `Retry-After` du provider est respecté, plafonné.
- **Circuit breaker** — « ce backend est-il en train de tomber ? » Après N échecs consécutifs,
  on cesse de l'appeler pendant un temps (**ouvert**), puis on laisse passer un appel d'essai
  (**semi-ouvert**) : succès → fermé, échec → rouvert. On protège le provider *et* nos appelants,
  qui échouent vite au lieu d'attendre.
- **Délai** — un provider muet est une panne, pas une attente.

Le temps et le hasard sont **injectés** : les tests jouent des heures en millisecondes, et ce
notebook n'attend jamais vraiment.

In [6]:
show_source(RetryPolicy.delay_for, CircuitBreaker.before_call, CircuitBreaker.record_failure)

policy = RetryPolicy(max_attempts=5, base_delay_s=0.5, max_delay_s=8.0, jitter=0.25)
rng = random.Random(7)
print("tentative | délai avant la suivante (backoff × gigue)")
for attempt in range(1, 6):
    print(f"    {attempt}     | {policy.delay_for(attempt, None, rng):6.2f} s")
print(
    f"avec Retry-After: 3 s → {policy.delay_for(1, 3.0, rng):.1f} s ; Retry-After: 60 s → plafonné à {policy.delay_for(1, 60.0, rng):.1f} s"
)

```python
    def delay_for(self, attempt: int, retry_after_s: float | None, rng: random.Random) -> float:
        """Délai avant la tentative ``attempt + 1`` (``attempt`` compte à partir de 1).

        Un ``Retry-After`` du provider est respecté, plafonné à ``max_delay_s`` : au-delà,
        on préfère réessayer tôt et échouer vite plutôt que bloquer un appelant.
        """
        if retry_after_s is not None:
            return min(max(retry_after_s, 0.0), self.max_delay_s)
        raw = self.base_delay_s * self.multiplier ** (attempt - 1)
        capped = min(raw, self.max_delay_s)
        return capped * (1.0 + rng.uniform(-self.jitter, self.jitter))
```
```python
    def before_call(self) -> None:
        state = self.state
        if state is CircuitState.OPEN:
            assert self._opened_at is not None
            remaining = self.recovery_timeout_s - (self._clock() - self._opened_at)
            raise CircuitOpenError(backend=self.backend, retry_in_s=max(remaining, 0.0))
        if state is CircuitState.HALF_OPEN:
            if self._trial_in_flight:
                raise CircuitOpenError(backend=self.backend, retry_in_s=0.0)
            self._trial_in_flight = True
```
```python
    def record_failure(self) -> None:
        self._consecutive_failures += 1
        if self._state is CircuitState.HALF_OPEN or (
            self._consecutive_failures >= self.failure_threshold
        ):
            self._state = CircuitState.OPEN
            self._opened_at = self._clock()
            self._trial_in_flight = False
```

tentative | délai avant la suivante (backoff × gigue)
    1     |   0.46 s
    2     |   0.83 s
    3     |   2.15 s
    4     |   3.14 s
    5     |   8.14 s
avec Retry-After: 3 s → 3.0 s ; Retry-After: 60 s → plafonné à 8.0 s


### 3.4 Budget, cache, gouvernance : refuser ou éviter *avant* d'appeler

- Le **budget** refuse un appel sur une estimation (quatre caractères par token, pessimiste) et
  réserve la sortie maximale : on refuse ce qui *pourrait* dépasser.
- Le **cache exact** a pour clé l'empreinte de tout ce qui influence la réponse — et de rien
  d'autre (les métadonnées de trace n'y entrent pas). Le **cache sémantique** compare l'embedding
  du message utilisateur dans un **espace de noms** qui fige tout le reste (modèle, system prompt,
  paramètres) : sans lui, « résume ce texte » servi à un autre system prompt renverrait la
  mauvaise réponse avec une similarité parfaite.
- La **politique de gouvernance** associe une classe de données à des exigences sur le backend
  (région d'inférence, rétention zéro, entraînement, sous-traitants). Une métadonnée inconnue
  échoue : l'inconnu n'est jamais un laissez-passer.

In [7]:
show_source(exact_key, semantic_namespace, GovernancePolicy.evaluate)

policy_cfg = GovernancePolicy.load(ROOT / "config" / "governance_policy.toml")
print("règles de la politique (config/governance_policy.toml) :")
for data_class, rule in policy_cfg.rules.items():
    print(
        f"  {data_class.value:<13} {rule.model_dump(exclude_defaults=True) or '(aucune exigence)'}"
    )

```python
def exact_key(request: CompletionRequest) -> str:
    return _digest(
        {
            "model": str(request.model),
            "messages": [(m.role.value, m.content) for m in request.messages],
            **_params(request),
        }
    )
```
```python
def semantic_namespace(request: CompletionRequest) -> str | None:
    """Espace de noms du cache sémantique, ou ``None`` si la requête n'y est pas éligible
    (plus d'un tour de conversation)."""
    turns = request.turns
    if len(turns) != 1 or turns[0].role is not Role.USER:
        return None
    return _digest({"model": str(request.model), "system": request.system, **_params(request)})
```
```python
    def evaluate(
        self, data_class: DataClass, backend: str, governance: BackendGovernance
    ) -> GovernanceDecision:
        rule = self.rules[data_class]
        reasons: list[str] = []

        region = governance.inference_region
        if rule.allowed_regions is not None and region not in rule.allowed_regions:
            allowed = ", ".join(r.value for r in rule.allowed_regions)
            reasons.append(f"région d'inférence {region.value!r} hors de [{allowed}]")
        if rule.require_zero_retention and governance.zero_retention is not True:
            state = "inconnue" if governance.zero_retention is None else "non configurée"
            reasons.append(f"rétention zéro exigée, {state}")
        if rule.forbid_training and governance.trains_on_inputs is not False:
            state = "inconnu" if governance.trains_on_inputs is None else "déclaré"
            reasons.append(f"entraînement sur les entrées interdit, {state}")
        if rule.forbid_variable_subprocessors and governance.subprocessors != "fixed":
            reasons.append("sous-traitants variables (courtier) interdits")

        return GovernanceDecision(
            data_class=data_class, backend=backend, allowed=not reasons, reasons=tuple(reasons)
        )
```

règles de la politique (config/governance_policy.toml) :
  public        (aucune exigence)
  internal      {'forbid_training': True}
  confidential  {'allowed_regions': (<InferenceRegion.EU: 'eu'>, <InferenceRegion.SELF: 'self'>), 'require_zero_retention': True, 'forbid_training': True, 'forbid_variable_subprocessors': True}
  personal      {'allowed_regions': (<InferenceRegion.EU: 'eu'>, <InferenceRegion.SELF: 'self'>), 'require_zero_retention': True, 'forbid_training': True, 'forbid_variable_subprocessors': True}
  regulated     {'allowed_regions': (<InferenceRegion.EU: 'eu'>, <InferenceRegion.SELF: 'self'>), 'require_zero_retention': True, 'forbid_training': True, 'forbid_variable_subprocessors': True}


### 3.5 La façade : l'ordre des maillons est le contrat

`LLMClient.complete` enchaîne, dans cet ordre : gouvernance → budget → cache → breaker → retry →
provider (avec délai) → budget → coût → trace. Lisez-la en entier une fois : tout ce que les
scénarios de la partie 4 vont montrer est dans ces quatre-vingts lignes.

In [8]:
show_source(LLMClient.complete)

```python
    async def complete(self, request: CompletionRequest, *, use_cache: bool = True) -> Completion:
        with self._tracer.start_as_current_span(f"chat {request.model}") as span:
            self._request_attributes(span, request, "chat")
            ref, config = self.registry.resolve(request.model)
            backend = ref.backend
            self._govern(span, request.data_class, backend, config)

            estimated = sum(estimate_tokens(m.content) for m in request.messages)
            if self.budget is not None:
                self.budget.check(
                    estimated_input_tokens=estimated, max_output_tokens=request.max_output_tokens
                )

            key = exact_key(request)
            namespace = semantic_namespace(request)
            vector: tuple[float, ...] | None = None
            if self.cache is not None and use_cache:
                hit, vector = await self._cache_lookup(span, request, key, namespace)
                if hit is not None:
                    cached = hit.model_copy(
                        update={"cached": True, "cost_usd": 0.0, "latency_ms": 0.0}
                    )
                    span.set_attribute(Bench.CACHED, True)
                    self._record_usage(span, Usage(), 0.0)
                    return cached
            span.set_attribute(Bench.CACHED, False)

            provider = self.provider_for(backend, config)
            completion, report = await self._guarded(
                backend, request.timeout_s, lambda: provider.complete(request)
            )

            if self.budget is not None:
                self.budget.record(completion.usage)
            cost = self._cost(completion.model, completion.usage, completion.usage_reported)
            completion = completion.model_copy(update={"cost_usd": cost, "attempts": report.count})

            span.set_attribute(GenAI.RESPONSE_MODEL, completion.model)
            span.set_attribute(GenAI.RESPONSE_FINISH_REASONS, [completion.finish_reason.value])
            if completion.provider_message_id:
                span.set_attribute(GenAI.RESPONSE_ID, completion.provider_message_id)
            span.set_attribute(Bench.LATENCY_MS, completion.latency_ms)
            span.set_attribute(Bench.ATTEMPTS, report.count)
            self._record_usage(span, completion.usage, cost)

            if self.cache is not None and use_cache:
                await self.cache.put(key, completion)
                if namespace is not None and vector is not None:
                    await self.cache.put_vector(namespace, vector, key)
            return completion
```

## Partie 4 — Exécution : faire tomber chaque maillon, et regarder

Le montage est le même pour tous les scénarios : un `LLMClient` construit sur la **vraie
configuration** du projet (`config/`), un exporteur de traces **en mémoire** pour lire les spans,
un `sleep` qui **enregistre** les délais au lieu de dormir, et un provider Anthropic réel branché sur
le réseau scripté. La fonction `scenario` ci-dessous n'est que de la glu de notebook.

In [9]:
class FakeClock:
    """Horloge injectée dans le breaker : on avance le temps à la main."""

    def __init__(self) -> None:
        self.now = 0.0

    def __call__(self) -> float:
        return self.now


def scenario(
    steps,
    *,
    attempts=None,
    breaker_threshold=None,
    recovery_s=30.0,
    clock=None,
    timeout_s=None,
    **extra,
):
    exporter = InMemorySpanExporter()
    tracer = get_tracer(configure_tracing(exporter=exporter, set_global=False))
    sleeps: list[float] = []

    async def recording_sleep(delay: float) -> None:
        sleeps.append(delay)

    client = LLMClient.from_settings(
        tracer=tracer,
        retry_policy=RetryPolicy(
            max_attempts=attempts or RETRY_MAX_ATTEMPTS, base_delay_s=0.5, jitter=0.0
        ),
        sleep=recording_sleep,
        breaker_factory=lambda backend: CircuitBreaker(
            backend=backend,
            failure_threshold=breaker_threshold or BREAKER_THRESHOLD,
            recovery_timeout_s=recovery_s,
            clock=clock or time.monotonic,
        ),
        default_timeout_s=timeout_s or TIMEOUT_S,
        **extra,
    )
    provider, script = scripted_anthropic_provider(steps, backend=ModelRef.parse(MODEL).backend)
    client.register_provider(ModelRef.parse(MODEL).backend, provider)
    return client, script, exporter, sleeps


def describe(completion) -> None:
    print(f"texte        : {completion.text!r}")
    print(f"modèle servi : {completion.model}  (demandé : {ModelRef.parse(MODEL).model})")
    print(f"usage        : {completion.usage.model_dump()}")
    print(
        f"coût         : {completion.cost_usd if completion.cost_usd is None else f'{completion.cost_usd:.6f} $'}"
    )
    print(
        f"tentatives   : {completion.attempts} — cache : {completion.cached} — fin : {completion.finish_reason.value}"
    )


print("montage prêt")

montage prêt


### 4.1 Chemin nominal : un appel, son coût, sa trace

Regardez la trace : les attributs `gen_ai.*` sont les **conventions sémantiques GenAI**
d'OpenTelemetry (le vocabulaire standard que Langfuse, Tempo ou Jaeger savent lire), les
`llm_testbench.*` ce que le banc ajoute — coût, cache, tentatives, gouvernance, empreinte du prompt.

In [10]:
client, script, exporter, sleeps = scenario(
    [Step(json_body=anthropic_message("Paris.", input_tokens=1_000, output_tokens=12))]
)
completion = await client.complete(ask("Quelle est la capitale de la France ?"))
describe(completion)
print()
show_span(exporter)

texte        : 'Paris.'
modèle servi : claude-sonnet-5  (demandé : claude-sonnet-5)
usage        : {'input_tokens': 1000, 'output_tokens': 12, 'cache_read_tokens': 0, 'cache_write_tokens': 0}
coût         : 0.002120 $
tentatives   : 1 — cache : False — fin : stop

span « chat anthropic/claude-sonnet-5 » — statut UNSET
  gen_ai.operation.name                          'chat'
  gen_ai.provider.name                           'anthropic'
  gen_ai.request.max_tokens                      200
  gen_ai.request.model                           'claude-sonnet-5'
  gen_ai.response.finish_reasons                 ('stop',)
  gen_ai.response.id                             'msg_scripted'
  gen_ai.response.model                          'claude-sonnet-5'
  gen_ai.usage.input_tokens                      1000
  gen_ai.usage.output_tokens                     12
  llm_testbench.attempts                         1
  llm_testbench.backend                          'anthropic'
  llm_testbench.cache.hit          

### 4.2 Limite de débit : 429 avec `Retry-After`, puis succès

Le provider dit « trop vite, revenez dans 2 s ». Notre retry respecte le délai conseillé (au lieu
de son propre backoff), puis réessaie. Deux tentatives, un seul sommeil, une seule facturation.

In [11]:
client, script, exporter, sleeps = scenario(
    [
        anthropic_error(429, "Too many requests", retry_after_s=2),
        Step(json_body=anthropic_message("Paris.")),
    ]
)
completion = await client.complete(ask("Capitale de la France ?"))
describe(completion)
print(f"\nrequêtes parties sur le réseau : {script.calls} — délais dormis (simulés) : {sleeps}")
show_span(exporter, prefixes=("llm_testbench.attempts",))

texte        : 'Paris.'
modèle servi : claude-sonnet-5  (demandé : claude-sonnet-5)
usage        : {'input_tokens': 120, 'output_tokens': 8, 'cache_read_tokens': 0, 'cache_write_tokens': 0}
coût         : 0.000320 $
tentatives   : 2 — cache : False — fin : stop

requêtes parties sur le réseau : 2 — délais dormis (simulés) : [2.0]
span « chat anthropic/claude-sonnet-5 » — statut UNSET
  llm_testbench.attempts                         2


### 4.3 Panne prolongée : le retry insiste, puis le breaker coupe

Le provider renvoie 529 en boucle. Avec `BREAKER_THRESHOLD` échecs consécutifs, le circuit
s'**ouvre** : la tentative suivante échoue *sans appel réseau*, et tout nouvel appel aussi, jusqu'à
la fin du délai de récupération. On avance ensuite l'horloge à la main pour voir le circuit passer
en **semi-ouvert** puis se refermer sur un succès.

In [12]:
clock = FakeClock()
client, script, exporter, sleeps = scenario(
    [anthropic_error(529, "Overloaded")], clock=clock, recovery_s=30.0
)

for attempt in (1, 2):
    try:
        await client.complete(ask("Capitale ?"))
    except LLMError as error:
        print(f"appel {attempt} → {type(error).__name__}: {error}")
    print(
        f"   requêtes réseau cumulées : {script.calls} — breaker : {client.breaker_for('anthropic').snapshot()}"
    )

clock.now += 31  # le délai de récupération est passé
client.register_provider(
    "anthropic", scripted_anthropic_provider([Step(json_body=anthropic_message("Paris."))])[0]
)
completion = await client.complete(ask("Capitale ?"))
print(
    f"\naprès 31 s simulées → appel d'essai réussi : {completion.text!r} — breaker : {client.breaker_for('anthropic').snapshot()}"
)

appel 1 → CircuitOpenError: circuit ouvert pour le backend 'anthropic', nouvel essai possible dans 30.0 s
   requêtes réseau cumulées : 2 — breaker : {'backend': 'anthropic', 'state': 'open', 'consecutive_failures': 2}
appel 2 → CircuitOpenError: circuit ouvert pour le backend 'anthropic', nouvel essai possible dans 30.0 s
   requêtes réseau cumulées : 2 — breaker : {'backend': 'anthropic', 'state': 'open', 'consecutive_failures': 2}

après 31 s simulées → appel d'essai réussi : 'Paris.' — breaker : {'backend': 'anthropic', 'state': 'closed', 'consecutive_failures': 0}


### 4.4 Le provider ne répond pas : le délai transforme l'attente en panne

La réponse arrive… en 0,5 s, pour un délai autorisé de `TIMEOUT_S`. Sans délai, l'appelant
attendrait indéfiniment ; avec, c'est une `ProviderTimeoutError`, **réessayable** (le provider n'a
peut-être rien reçu). Ici on limite à une tentative pour voir l'erreur nue.

In [13]:
client, script, exporter, sleeps = scenario(
    [Step(delay_s=0.5, json_body=anthropic_message("Paris."))], attempts=1
)
started = time.perf_counter()
try:
    await client.complete(ask("Capitale ?"))
except ProviderTimeoutError as error:
    print(
        f"{type(error).__name__} après {time.perf_counter() - started:.2f} s : {error} — réessayable : {error.retryable}"
    )

ProviderTimeoutError après 0.20 s : aucune réponse de 'anthropic' en 0.2 s — réessayable : True


### 4.5 Refuser avant d'appeler : gouvernance et budget

Une requête étiquetée `regulated` ne peut pas partir vers un backend américain sans rétention
zéro (politique provisoire, `config/governance_policy.toml`). Le refus est une exception typée
**avant tout appel réseau** (comptez les requêtes : zéro), et la trace garde la décision et ses
raisons. Même logique pour le budget : l'estimation refuse, le provider n'est jamais sollicité.

In [14]:
client, script, exporter, sleeps = scenario([Step(json_body=anthropic_message("Paris."))])
try:
    await client.complete(ask("Dossier patient n° 42 ?", data_class="regulated"))
except GovernanceViolation as error:
    print(f"GovernanceViolation : {error}")
print(f"requêtes réseau : {script.calls}")
show_span(exporter, prefixes=("llm_testbench.governance",))

tiny_budget = TokenBudget(max_input_tokens_per_request=10)
client, script, exporter, sleeps = scenario(
    [Step(json_body=anthropic_message("Paris."))], budget=tiny_budget
)
try:
    await client.complete(ask("Une question bien trop longue pour un budget de dix tokens ?"))
except BudgetExceededError as error:
    print(f"\nBudgetExceededError : {error} — requêtes réseau : {script.calls}")

GovernanceViolation : backend 'anthropic' refusé pour la classe de données 'regulated' : région d'inférence 'us' hors de [eu, self] ; rétention zéro exigée, non configurée
requêtes réseau : 0
span « chat anthropic/claude-sonnet-5 » — statut ERROR
  llm_testbench.governance.allowed               False
  llm_testbench.governance.data_class            'regulated'
  llm_testbench.governance.inference_region      'us'
  llm_testbench.governance.reasons               ("région d'inférence 'us' hors de [eu, self]", 'rétention zéro exigée, non configurée')

BudgetExceededError : entrée estimée à 21 tokens, plafond par requête 10 — requêtes réseau : 0


### 4.6 Les caches : exact, puis sémantique

Trois appels. Le deuxième est la copie exacte du premier : servi depuis le cache exact, coût nul,
aucune requête réseau. Le troisième reformule la question : le cache exact rate, le cache
**sémantique** compare les embeddings (ici un embedder jouet à trois dimensions, importé de
`src/`) et sert la réponse si la similarité dépasse `SEMANTIC_THRESHOLD`. Regardez le compteur
de requêtes réseau et l'attribut `cache.kind` des traces.

In [15]:
client, script, exporter, sleeps = scenario(
    [Step(json_body=anthropic_message("Paris.", input_tokens=800, output_tokens=6))],
    cache=InMemoryCache(),
    semantic_cache=SemanticCacheConfig(
        embedding_model=ModelRef.parse("openai/text-embedding-3-small"),
        min_similarity=SEMANTIC_THRESHOLD,
    ),
)
client.register_embedder("openai", ToyEmbedder())

for question in (
    "Quelle est la capitale de la France ?",
    "Quelle est la capitale de la France ?",
    "Capitale de la France, s'il te plaît ?",
):
    completion = await client.complete(ask(question))
    attrs = dict(exporter.get_finished_spans()[-1].attributes or {})
    print(
        f"{question:<42} cache={completion.cached!s:<5} kind={attrs.get('llm_testbench.cache.kind', '-'):<8} coût={completion.cost_usd:.6f} $  réseau={script.calls}"
    )
    exporter.clear()

Quelle est la capitale de la France ?      cache=False kind=-        coût=0.001660 $  réseau=1
Quelle est la capitale de la France ?      cache=True  kind=exact    coût=0.000000 $  réseau=1
Capitale de la France, s'il te plaît ?     cache=True  kind=semantic coût=0.000000 $  réseau=1


### 4.7 Sortie structurée : demander un schéma, valider, réparer

On veut un objet Pydantic, pas du texte. Le backend Anthropic sait **imposer** un schéma JSON
(`output_config.format`) ; on le voit dans les paramètres envoyés. Mais un schéma JSON ne sait pas
tout exprimer (ici, une confiance entre 0 et 1) : la validation Pydantic reste, et quand elle
échoue, la couche **renvoie l'erreur au modèle** dans un message de réparation — au plus
`max_repairs` fois. Regardez le message de réparation exact : c'est le prompt versionné
`prompts/structured_output_repair.md`, rendu.

In [16]:
from pydantic import BaseModel, Field


class Capital(BaseModel):
    country: str
    city: str
    confidence: float = Field(ge=0.0, le=1.0)


client, script, exporter, sleeps = scenario(
    [
        Step(
            json_body=anthropic_message('{"country": "France", "city": "Paris", "confidence": 7}')
        ),
        Step(
            json_body=anthropic_message(
                '{"country": "France", "city": "Paris", "confidence": 0.98}'
            )
        ),
    ]
)
structured = StructuredOutputs(client, PromptRegistry.from_settings(), max_repairs=2)
result = await structured.complete(ask("Capitale de la France, avec ta confiance ?"), Capital)

print(f"valeur validée : {result.value!r} — tentatives : {result.attempts}")
print(
    f"\nschéma imposé au provider (output_config) : {json.dumps(script.requests[0]['output_config'], ensure_ascii=False)[:160]}…"
)
print(f"\nerreur de validation renvoyée au modèle :\n{result.repairs[0][:300]}")
print("\nmessage de réparation réellement envoyé (dernier message de la 2e requête) :")
print(script.requests[1]["messages"][-1]["content"])

valeur validée : Capital(country='France', city='Paris', confidence=0.98) — tentatives : 2

schéma imposé au provider (output_config) : {"format": {"type": "json_schema", "schema": {"properties": {"country": {"title": "Country", "type": "string"}, "city": {"title": "City", "type": "string"}, "co…

erreur de validation renvoyée au modèle :
1 validation error for Capital
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=7, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

message de réparation réellement envoyé (dernier message de la 2e requête) :
Ta réponse précédente n'est pas conforme au schéma attendu. Erreur de validation :

1 validation error for Capital
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=7, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

Renvoie uniquement l'objet JSON corrigé, complet, 

### 4.8 Streaming : les tokens arrivent au fil de l'eau, l'usage à la fin

En **streaming** (SSE), le provider envoie la réponse morceau par morceau : l'utilisateur voit le
premier mot après quelques centaines de millisecondes (le **TTFT**, time to first token) au lieu
d'attendre la phrase entière. Le prix : on ne peut pas **réessayer** un flux entamé sans dupliquer
du texte déjà affiché — la façade ne retente donc pas après le premier morceau — et l'usage
n'est connu qu'à la fin. Le coût est calculé sur le dernier morceau.

In [17]:
client, script, exporter, sleeps = scenario(
    [Step(sse_body=anthropic_sse(["La ", "capitale ", "est ", "Paris."]))]
)
started = time.perf_counter()
async for chunk in client.stream(ask("Capitale ?")):
    elapsed = (time.perf_counter() - started) * 1000
    if chunk.kind == "text":
        print(f"  +{elapsed:5.1f} ms  texte  {chunk.text!r}")
    elif chunk.kind == "usage":
        print(
            f"  +{elapsed:5.1f} ms  usage  {chunk.usage.model_dump()}  coût={chunk.cost_usd:.6f} $"
        )
    else:
        print(f"  +{elapsed:5.1f} ms  fin    {chunk.finish_reason.value}")

  + 17.8 ms  texte  'La '
  + 17.9 ms  texte  'capitale '
  + 18.0 ms  texte  'est '
  + 18.0 ms  texte  'Paris.'
  + 21.4 ms  usage  {'input_tokens': 120, 'output_tokens': 8, 'cache_read_tokens': 0, 'cache_write_tokens': 0}  coût=0.000320 $
  + 21.5 ms  fin    stop


### 4.9 Avec un vrai provider (mode complet) : le prompt caching en chiffres

Cette cellule ne s'exécute qu'en mode complet avec `ANTHROPIC_API_KEY`. Elle envoie deux fois le
même appel avec un **system prompt long** (au-delà du minimum cacheable du modèle) : au premier
appel, les tokens du system prompt sont **écrits** dans le cache (facturés 1,25×) ; au second,
**lus** (0,1×). Comparez `cache_write_tokens`, `cache_read_tokens` et le coût des deux appels :
c'est la différence entre un RAG qui coûte et un RAG qui coûte dix fois moins sur son préfixe fixe.

In [18]:
if not USE_REAL_API:
    print("section sautée : mode échantillon ou clé ANTHROPIC_API_KEY absente (voir .env.example)")
else:
    exporter = InMemorySpanExporter()
    real = LLMClient.from_settings(
        tracer=get_tracer(configure_tracing(exporter=exporter, set_global=False))
    )
    paragraph = (
        "Tu es l'assistant d'un banc d'essai de techniques LLM. Tu réponds en français, brièvement, "
        "et tu cites tes sources quand on t'en donne. Ce paragraphe est répété pour dépasser la taille "
        "minimale de mise en cache du préfixe du prompt. "
    )
    long_system = paragraph * 60  # ≈ 2 500 tokens
    for label in ("1er appel", "2e appel"):
        completion = await real.complete(
            ask(
                "Réponds simplement : quelle est la capitale de la France ?",
                system=long_system,
                max_output_tokens=30,
            ),
            use_cache=False,
        )
        u = completion.usage
        print(
            f"{label} : modèle servi={completion.model} entrée={u.input_tokens} cache_write={u.cache_write_tokens} cache_read={u.cache_read_tokens} sortie={u.output_tokens} coût={completion.cost_usd:.5f} $ latence={completion.latency_ms:.0f} ms"
        )

section sautée : mode échantillon ou clé ANTHROPIC_API_KEY absente (voir .env.example)


## Partie 5 — Les pièges (ce que les tutoriels ne disent pas)

1. **Deux couches de retry** : le SDK retente déjà 429 et 5xx (deux fois par défaut). Empiler
   notre retry par-dessus multiplie les tentatives sans que personne ne le voie. On a mis le SDK à
   zéro : une seule politique, la nôtre, observable.
2. **`temperature=0` renvoie un 400** sur les modèles Claude actuels. Le déterminisme se *mesure*
   (phase 5B), il ne se décrète plus par un paramètre.
3. **`Retry-After` peut dire 60 s.** Le respecter à la lettre bloque un appelant une minute ; on
   le plafonne, et on préfère échouer vite.
4. **Le cache sémantique ment avec confiance** : deux questions proches au sens de l'embedding
   peuvent attendre des réponses différentes (« capitale de la France » / « capitale de la
   Finlande » selon le modèle d'embeddings). D'où un seuil élevé, un espace de noms strict, et
   une mesure du taux de faux positifs avant de l'activer en production (phase 5).
5. **Un coût inconnu n'est pas un coût nul.** Un modèle absent de la table de prix donne
   `None`, et la trace le dit (`cost_known=false`). Un zéro silencieux se propagerait jusqu'au
   tableau maître.
6. **Le modèle servi n'est pas forcément le modèle demandé** : la réponse porte l'identifiant
   exact, on le trace. Un provider qui change de révision sans prévenir est le sujet d'un runbook
   de la phase 5.
7. **Le prompt caching a une taille minimale** (de l'ordre du millier de tokens selon le modèle) :
   sur un prompt court, `cache_read_tokens` reste à zéro, et ce n'est pas un bug.
8. **Un flux entamé ne se réessaie pas.** Si le provider tombe au milieu, l'utilisateur a déjà vu
   la moitié de la réponse : à l'appelant de décider (recommencer, afficher l'erreur), pas à la
   couche de dupliquer du texte.
9. **Les métadonnées de gouvernance sont des affirmations**, datées, relevées sur des pages
   publiques : elles se vérifient contractuellement, elles ne se supposent pas.

## Partie 6 — À vous de jouer

Modifiez la configuration en haut, puis « Run All » :

1. **`RETRY_MAX_ATTEMPTS = 1`** — le scénario 4.2 échoue au premier 429 ; à `5`, le scénario 4.3
   insiste davantage avant que le breaker ne coupe (comptez les requêtes réseau).
2. **`BREAKER_THRESHOLD = 10`** — en 4.3, le breaker ne s'ouvre plus : chaque appel paie toutes
   ses tentatives. C'est le coût d'un breaker trop tolérant.
3. **`TIMEOUT_S = 1.0`** — en 4.4, la réponse lente passe : à vous de dire si 0,5 s d'attente est
   acceptable pour votre cas d'usage.
4. **`SEMANTIC_THRESHOLD = 0.5`** — en 4.6, la reformulation est servie… et une question sur une
   recette le serait peut-être aussi : ajoutez-la à la liste et regardez.
5. **`DATA_CLASS = "regulated"`** — tous les scénarios refusent avant d'appeler. Puis, dans
   `config/governance_policy.toml`, autorisez `us` pour `regulated` et relancez : la politique vit
   dans la configuration, pas dans le code.
6. **`MODEL = "openai/gpt-5.6-luna"`** — les scénarios simulés restent sur le provider Anthropic
   (le script est un script Anthropic), mais les coûts de la partie 7 changent.

## Partie 7 — Les chiffres

Ce notebook mesure des **mécanismes**, pas une qualité : il n'y a pas de score de littérature à
comparer ici (ça commence au notebook 2 pour les embeddings, et en phase 2 pour tout le reste).
Ce qu'on peut chiffrer dès maintenant, c'est **ce que coûte un appel** selon le modèle, et ce que
ça devient à l'échelle.

In [19]:
pricing = LLMClient.from_settings().pricing
typical = Usage(
    input_tokens=1_000, output_tokens=300
)  # un appel RAG typique : contexte + réponse courte
cached = Usage(
    input_tokens=200, cache_read_tokens=800, output_tokens=300
)  # le même, préfixe servi par le cache

print(f"table de prix vérifiée le {pricing.verified_on} — {pricing.source}\n")
print(
    f"{'modèle':<24} {'1 appel':>10} {'1 appel (cache)':>16} {'100 k appels/mois':>18} {'cache':>10}"
)
for model in sorted(pricing.models):
    if pricing.models[model].output_per_mtok == 0:
        continue  # embeddings : pas de sortie
    one = pricing.cost_usd(model, typical)
    with_cache = pricing.cost_usd(model, cached)
    monthly = one * 100_000
    saving = f"−{(1 - with_cache / one) * 100:.0f} %" if with_cache is not None else "n/a"
    print(f"{model:<24} {one:>9.5f} $ {with_cache:>14.5f} $ {monthly:>16,.0f} $ {saving:>10}")

table de prix vérifiée le 2026-09-03 — Anthropic : référence API du 2026-06-24 (cache : lecture 0,1x, écriture 1,25x) ; OpenAI : page de prix officielle relevée le 2026-09-03

modèle                      1 appel  1 appel (cache)  100 k appels/mois      cache
claude-haiku-4-5           0.00250 $        0.00178 $              250 $      −29 %
claude-opus-5              0.01250 $        0.00890 $            1,250 $      −29 %
claude-sonnet-5            0.00500 $        0.00356 $              500 $      −29 %
gpt-5-mini                 0.00085 $        0.00067 $               85 $      −21 %
gpt-5-nano                 0.00017 $        0.00013 $               17 $      −21 %
gpt-5.4-mini               0.00210 $        0.00156 $              210 $      −26 %
gpt-5.4-nano               0.00057 $        0.00043 $               58 $      −25 %
gpt-5.6-luna               0.00056 $        0.00042 $               56 $      −26 %
gpt-5.6-sol                0.01000 $        0.00712 $            1,00

## Questions d'entretien

<details><summary><b>1. Quelle différence entre un retry et un circuit breaker ? Pourquoi les deux ?</b></summary>

Le retry raisonne par appel : « cette tentative a échoué, la suivante peut réussir » (429, 5xx,
coupure). Le breaker raisonne par backend : « il tombe, arrêtons de l'appeler ». Sans breaker, une
panne prolongée fait payer à chaque appelant toutes ses tentatives et bombarde un service déjà à
terre ; sans retry, un 429 isolé devient une erreur utilisateur. Le breaker ne compte que les
erreurs qui parlent de la santé du backend (indisponible, délai) — pas les 400 (notre faute) ni
les 429 (une limite, pas une panne).
</details>

<details><summary><b>2. Pourquoi de la gigue dans le backoff ?</b></summary>

Après une panne, tous les clients qui ont échoué à la même seconde réessaieraient à la même
seconde : une vague synchronisée qui reproduit la surcharge (le « thundering herd »). Une gigue
aléatoire de ±25 % étale la vague.
</details>

<details><summary><b>3. Comment garantir qu'aucune donnée sensible ne part vers le mauvais provider ?</b></summary>

Pas par une règle d'équipe : par construction. Chaque requête porte une classe de données
obligatoire ; chaque backend déclare des métadonnées de gouvernance datées ; une politique
(fichier de configuration) décide par refus par défaut, avant tout appel réseau, et la décision
est tracée. Les embeddings et le juge passent par la même politique : le texte sort par ces
chemins-là aussi.
</details>

<details><summary><b>4. Que met-on dans la clé d'un cache exact, et qu'est-ce qui rend un cache sémantique dangereux ?</b></summary>

Tout ce qui influence la réponse (backend, modèle, messages, paramètres, schéma) et rien d'autre
(pas les métadonnées de trace). Le cache sémantique est dangereux parce qu'il sert une réponse à
une question *proche*, pas identique : il faut un espace de noms qui fige le contexte (system
prompt, modèle, paramètres), un seuil élevé, l'exclure des conversations à historique, et mesurer
son taux de faux positifs.
</details>

<details><summary><b>5. Pourquoi ne pas retenter un appel en streaming ?</b></summary>

Parce que des tokens ont déjà été envoyés au client. Rejouer l'appel produirait un texte différent
(non-déterminisme) et dupliquerait le début. On retente avant le premier morceau ; après, on
signale l'erreur et on laisse l'appelant décider. Le breaker, lui, est bien alimenté par l'échec.
</details>

<details><summary><b>6. Comment calculez-vous le coût d'un appel, et que faites-vous d'un prix inconnu ?</b></summary>

Usage normalisé (tokens d'entrée non cachés, tokens de sortie, lectures et écritures de cache)
multiplié par une table de prix versionnée et datée. Un modèle absent de la table donne un coût
inconnu (`None`), tracé comme tel ; jamais un zéro qui fausserait un tableau plus loin.
</details>

## Ce qu'on n'a pas fait, et pourquoi

- **Pas d'API batch** (moitié prix, résultats sous 24 h) : elle arrive avec le harness de la
  phase 2, quand des milliers d'appels de juge la justifieront.
- **Pas de repli automatique vers un autre backend** en cas de panne : le repli est une décision
  de gouvernance (quel backend a le droit de recevoir cette requête ?), donc de l'appelant, tracée.
- **Budget par processus** : le compteur est en mémoire ; la phase 5 le partagera (Postgres) entre
  réplicas de l'API.
- **Cache sémantique en mémoire ici** : la version Postgres/pgvector arrive avec l'ingestion
  (notebook 2), sur la même base.
- **Pas de backend de traces branché** : les spans sont exportés en mémoire pour être lus ici ;
  Langfuse les reçoit en OTLP (guide infra 07).

**La suite** : notebook 2, du document au chunk indexé — et la première mesure du banc (trois
modèles d'embeddings sur SciFact, comparés à la littérature).

## Lexique récapitulatif

| Terme | Définition |
|---|---|
| **Provider** | Le service distant qui exécute le modèle (Anthropic, OpenAI, OpenRouter, Bedrock, un vLLM à nous). |
| **Backend** | Une entrée de `config/llm_backends.toml` : un provider configuré, avec ses métadonnées de gouvernance. |
| **Token** | L'unité de texte facturée (~4 caractères en anglais). |
| **Complétion** | La réponse générée par le modèle. |
| **TTFT** | Time to first token : délai avant le premier morceau en streaming. |
| **Backoff exponentiel** | Délai entre tentatives qui double à chaque échec, plafonné. |
| **Gigue (jitter)** | Part aléatoire du délai, pour désynchroniser les clients. |
| **Circuit breaker** | Compteur d'échecs par backend qui coupe les appels pendant un temps (fermé / ouvert / semi-ouvert). |
| **Retry-After** | En-tête HTTP par lequel un provider indique quand réessayer. |
| **Raison de fin** | Pourquoi le modèle s'est arrêté : `stop` (fini), `length` (tronqué), `refusal`… |
| **Sortie structurée** | Réponse contrainte à un schéma JSON, validée par un modèle Pydantic. |
| **Cache exact / sémantique** | Réponse réutilisée pour une requête identique / pour une question similaire au sens de l'embedding. |
| **Embedding** | Vecteur numérique représentant le sens d'un texte ; deux textes proches ont des vecteurs proches. |
| **Classe de données** | Étiquette de sensibilité d'une requête (`public` … `regulated`), obligatoire. |
| **Prompt caching** | Réutilisation par le provider d'un préfixe de prompt identique, facturé beaucoup moins cher. |
| **Conventions sémantiques GenAI** | Noms d'attributs standard d'OpenTelemetry pour les appels LLM (`gen_ai.*`). |
| **Span** | L'unité d'une trace : une opération, sa durée, ses attributs. |